# Лабораторная работа №5. Работа с файлами в Linux средствами ассемблера

**Выполнил:** Рулев Константин Александрович  
**Вариант:** 16

**Цель:** научиться читать, изменять и записывать текстовые файлы на ассемблере.

Отчёт содержит исходный код, сборку и проверки. Для запуска в Google Colab выберите обычную CPU-среду и выполните все ячейки по порядку.

Результаты получены при выполнении в Linux x86-64. Все файлы для повторного запуска встроены в отчёт.


## 1. Задания варианта

**Задание 6.** Удвоить все согласные латинские буквы обоих регистров и сохранить результат во второй файл. Буквы `Y` и `y` тоже удваиваются.

**Задание 8.** Записать в третий файл символы, которые встречаются в обоих исходных файлах. Каждый общий символ записывается один раз, в порядке первого появления в первом файле. Регистр учитывается. Пробелы и переводы строк также являются символами.

In [1]:
import random

r = random.Random(16)
print("Задания:", [i + 1 for i in r.sample(range(11), 2)])

Задания: [6, 8]


## 2. Алгоритмы

В задании 6 текст читается блоками по 4096 байт. Каждая буква проверяется: согласная записывается дважды, остальные символы — один раз.

В задании 8 сначала запоминаются символы второго файла. Затем читается первый файл и записываются общие символы без повторов.

Имена файлов передаются в командной строке. Выходной файл должен быть новым: если он уже существует, программа завершится с ошибкой и сохранит его содержимое. Размер входных файлов не ограничен размером буфера.

## 3. Подготовка среды

In [2]:
import os
import platform
import shutil
import subprocess
from pathlib import Path

if platform.system() != "Linux" or platform.machine() not in ("x86_64", "amd64"):
    raise RuntimeError("Нужна CPU-среда Linux x86-64")
if not all(shutil.which(name) for name in ("fasm", "ld", "make")):
    prefix = [] if os.geteuid() == 0 else ["sudo"]
    subprocess.run(prefix + ["apt-get", "update", "-qq"], check=True)
    subprocess.run(prefix + ["apt-get", "install", "-y", "-qq", "fasm", "binutils", "make"], check=True)
base = Path("/content") if Path("/content").exists() else Path("/tmp")
work = base / "rulev_variant16" / "Lab_5"
work.mkdir(parents=True, exist_ok=True)
(work / "examples").mkdir(exist_ok=True)
os.chdir(work)
print("Среда готова")

Среда готова


## 4. Исходный код

### task6.asm

In [3]:
%%writefile task6.asm
format ELF64
public _start

section '.bss' writable
input_buffer rb 4096
output_buffer rb 8192

section '.text' executable
_start:
    cmp qword [rsp], 3
    jne error
    mov eax, 2
    mov rdi, [rsp + 16]
    xor esi, esi
    syscall
    test rax, rax
    js error
    mov r12, rax
    mov eax, 2
    mov rdi, [rsp + 24]
    mov esi, 193
    mov edx, 644o
    syscall
    test rax, rax
    js error
    mov r13, rax
.read:
    xor eax, eax
    mov rdi, r12
    mov rsi, input_buffer
    mov edx, 4096
    syscall
    cmp rax, -4
    je .read
    test rax, rax
    js error
    jz .close
    mov r8, rax
    xor r9d, r9d
    xor r10d, r10d
.letter:
    mov al, [input_buffer + r9]
    mov [output_buffer + r10], al
    inc r10
    mov bl, al
    and bl, 0DFh
    cmp bl, 'A'
    jb .next
    cmp bl, 'Z'
    ja .next
    cmp bl, 'A'
    je .next
    cmp bl, 'E'
    je .next
    cmp bl, 'I'
    je .next
    cmp bl, 'O'
    je .next
    cmp bl, 'U'
    je .next
    mov [output_buffer + r10], al
    inc r10
.next:
    inc r9
    cmp r9, r8
    jb .letter
    mov rdi, r13
    mov rsi, output_buffer
    mov rdx, r10
    call write_all
    jmp .read
.close:
    mov eax, 3
    mov rdi, r12
    syscall
    test rax, rax
    js error
    mov eax, 3
    mov rdi, r13
    syscall
    test rax, rax
    js error
    jmp finish

include 'io.inc'


Writing task6.asm


### task8.asm

In [4]:
%%writefile task8.asm
format ELF64
public _start

section '.bss' writable
input_buffer rb 4096
output_buffer rb 256
present rb 256

section '.text' executable
_start:
    cmp qword [rsp], 4
    jne error
    mov eax, 2
    mov rdi, [rsp + 16]
    xor esi, esi
    syscall
    test rax, rax
    js error
    mov r12, rax
    mov eax, 2
    mov rdi, [rsp + 24]
    xor esi, esi
    syscall
    test rax, rax
    js error
    mov r13, rax
    mov eax, 2
    mov rdi, [rsp + 32]
    mov esi, 193
    mov edx, 644o
    syscall
    test rax, rax
    js error
    mov r14, rax
.read_second:
    xor eax, eax
    mov rdi, r13
    mov rsi, input_buffer
    mov edx, 4096
    syscall
    cmp rax, -4
    je .read_second
    test rax, rax
    js error
    jz .read_first
    mov r8, rax
    xor r9d, r9d
.mark:
    movzx ebx, byte [input_buffer + r9]
    mov byte [present + rbx], 1
    inc r9
    cmp r9, r8
    jb .mark
    jmp .read_second
.read_first:
    xor eax, eax
    mov rdi, r12
    mov rsi, input_buffer
    mov edx, 4096
    syscall
    cmp rax, -4
    je .read_first
    test rax, rax
    js error
    jz .close
    mov r8, rax
    xor r9d, r9d
    xor r10d, r10d
.select:
    movzx ebx, byte [input_buffer + r9]
    cmp byte [present + rbx], 0
    je .next
    mov [output_buffer + r10], bl
    inc r10
    mov byte [present + rbx], 0
.next:
    inc r9
    cmp r9, r8
    jb .select
    mov rdi, r14
    mov rsi, output_buffer
    mov rdx, r10
    call write_all
    jmp .read_first
.close:
    mov eax, 3
    mov rdi, r12
    syscall
    test rax, rax
    js error
    mov eax, 3
    mov rdi, r13
    syscall
    test rax, rax
    js error
    mov eax, 3
    mov rdi, r14
    syscall
    test rax, rax
    js error
    jmp finish

include 'io.inc'


Writing task8.asm


### io.inc

In [5]:
%%writefile io.inc
section '.text' executable

write_all:
    test rdx, rdx
    jz .done
.next:
    mov eax, 1
    syscall
    cmp rax, -4
    je .next
    test rax, rax
    jle error
    add rsi, rax
    sub rdx, rax
    jnz .next
.done:
    ret

error:
    mov eax, 1
    mov edi, 2
    mov rsi, error_text
    mov edx, error_length
    syscall
    mov eax, 60
    mov edi, 1
    syscall

finish:
    mov eax, 60
    xor edi, edi
    syscall

section '.data' writable
error_text db 'Error: check arguments, input files and a NEW output filename.', 10
error_length = $ - error_text


Writing io.inc


### Makefile

In [6]:
%%writefile Makefile
.PHONY: all check clean

all: build/task6 build/task8

build:
	mkdir -p build

build/%.o: %.asm io.inc | build
	fasm $< $@

build/task%: build/task%.o
	ld -z noexecstack -o $@ $<

check: all
	python3 check.py

clean:
	rm -rf build


Writing Makefile


## 5. Сборка

In [7]:
%%bash
set -e
make all

mkdir -p build
fasm task6.asm build/task6.o
flat assembler  version 1.73.32  (16384 kilobytes memory)
4 passes, 1592 bytes.
ld -z noexecstack -o build/task6 build/task6.o
fasm task8.asm build/task8.o
flat assembler  version 1.73.32  (16384 kilobytes memory)
4 passes, 1840 bytes.
ld -z noexecstack -o build/task8 build/task8.o
rm build/task8.o build/task6.o


## 6. Примеры выполнения

В первом примере удваиваются согласные. Во втором общие символы — `a`, `b`, `r`, `c` и перевод строки.

In [8]:
%%writefile examples/text.txt
Hello, World! Yy AEIOU aeiou 123.


Writing examples/text.txt


In [9]:
%%writefile examples/first.txt
abracadabra


Writing examples/first.txt


In [10]:
%%writefile examples/second.txt
barbecue


Writing examples/second.txt


In [11]:
from pathlib import Path
import subprocess

for filename in ("build/doubled.txt", "build/common.txt"):
    Path(filename).unlink(missing_ok=True)
subprocess.run(["./build/task6", "examples/text.txt", "build/doubled.txt"], check=True)
print("Исходный текст:")
print(Path("examples/text.txt").read_text(), end="")
print("Результат задания 6:")
print(Path("build/doubled.txt").read_text(), end="")
subprocess.run(["./build/task8", "examples/first.txt", "examples/second.txt", "build/common.txt"], check=True)
print("Первый файл:", repr(Path("examples/first.txt").read_text()))
print("Второй файл:", repr(Path("examples/second.txt").read_text()))
print("Результат задания 8:", repr(Path("build/common.txt").read_text()))

Исходный текст:
Hello, World! Yy AEIOU aeiou 123.
Результат задания 6:
HHellllo, WWorrlldd! YYyy AEIOU aeiou 123.
Первый файл: 'abracadabra\n'
Второй файл: 'barbecue\n'
Результат задания 8: 'abrc\n'


## 7. Проверка

Проверяются пустые файлы, буквы обоих регистров, повторы, разные символы, тексты больше одного блока и ошибки имён файлов. Отдельно проверяется сохранность уже существующих файлов.

In [12]:
%%writefile check.py
from pathlib import Path
import os
import random
import subprocess
import tempfile

root = Path(__file__).resolve().parent
consonants = set(b'BCDFGHJKLMNPQRSTVWXYZbcdfghjklmnpqrstvwxyz')
count = 0


def run(name, arguments, success=True):
    global count
    result = subprocess.run([str(root / 'build' / name), *map(str, arguments)], capture_output=True, timeout=10)
    assert result.returncode == (0 if success else 1), (name, result.returncode, result.stderr)
    count += 1
    return result


with tempfile.TemporaryDirectory() as directory:
    work = Path(directory)
    first = work / 'first.txt'
    second = work / 'second.txt'
    output = work / 'output.txt'
    rng = random.Random(16)
    alphabet = b'ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789 \n\t.,!?'
    samples = [b'', b'AEIOUaeiou', b'Yy', b'Hello, World! 123\n', bytes(range(256)), b'B' * 8193]
    samples += [bytes(rng.choice(alphabet) for _ in range(size)) for size in (1, 17, 4095, 4096, 4097, 17000)]
    for data in samples:
        first.write_bytes(data)
        output.unlink(missing_ok=True)
        run('task6', [first, output])
        expected = b''.join(bytes([value]) * (2 if value in consonants else 1) for value in data)
        assert output.read_bytes() == expected
        assert first.read_bytes() == data

    pairs = [(b'', b''), (b'abc', b''), (b'', b'abc'), (b'aaaa', b'aaa'), (b'abc', b'XYZ'), (b'abracadabra\n', b'barbecue\n'), (b' \n\t!11', b'1!\t\n '), (bytes(range(256)) * 33, bytes(range(255, -1, -1)))]
    pairs += [(bytes(rng.choice(alphabet) for _ in range(size)), bytes(rng.choice(alphabet) for _ in range(size + 7))) for size in (1, 4095, 4096, 4097, 17000)]
    for a, b in pairs:
        first.write_bytes(a)
        second.write_bytes(b)
        output.unlink(missing_ok=True)
        run('task8', [first, second, output])
        expected = bytes(dict.fromkeys(value for value in a if value in set(b)))
        assert output.read_bytes() == expected
        assert first.read_bytes() == a
        assert second.read_bytes() == b

    for name, arguments in [('task6', []), ('task6', [first]), ('task6', [first, second, output]), ('task8', []), ('task8', [first, second]), ('task8', [first, second, output, output])]:
        run(name, arguments, False)

    first.write_bytes(b'Hello, World!')
    second.write_bytes(b'World')
    for name in ('task6', 'task8'):
        inputs = [first] if name == 'task6' else [first, second]
        output.unlink(missing_ok=True)
        run(name, [work / 'missing.txt', *inputs[1:], output], False)
        assert not output.exists()
        run(name, [*inputs, work / 'missing' / 'out.txt'], False)
        output.write_bytes(b'Keep this data')
        run(name, [*inputs, output], False)
        assert output.read_bytes() == b'Keep this data'
        run(name, [*inputs, first], False)
        assert first.read_bytes() == b'Hello, World!'
        link = work / 'link.txt'
        os.link(first, link)
        run(name, [*inputs, link], False)
        assert first.read_bytes() == b'Hello, World!'
        link.unlink()
        link.symlink_to(first)
        run(name, [*inputs, link], False)
        assert first.read_bytes() == b'Hello, World!'
        link.unlink()

    run('task8', [first, second, second], False)
    assert second.read_bytes() == b'World'
    output.unlink(missing_ok=True)
    run('task8', [first, work / 'missing.txt', output], False)
    assert not output.exists()

print(f'Проверок пройдено: {count}. Результаты обеих программ верны.')


Writing check.py


In [13]:
%%bash
set -e
make check

python3 check.py
Проверок пройдено: 45. Результаты обеих программ верны.


## Вывод

Написаны две программы на ассемблере: удвоение согласных и поиск общих символов файлов. Для работы с файлами использованы системные вызовы открытия, чтения, записи и закрытия. Результаты проверяются на обычных и граничных данных.